# Supplied-context development diagnostic
207 AI-reviewed dev excerpts; fixed text-only RoBERTa vs title-only Qwen vs title+plot Qwen. 94 have summaries; others reuse title-only predictions. No training, API, purchases or test tuning. Existing Colab GPU units only. Download results before deleting runtime. Context challenge shares source plots; report separately from real content.


In [ ]:
import base64, hashlib, io, zipfile, subprocess, sys, json, shutil
from pathlib import Path
from IPython.display import HTML, display
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='ab02957d3276b46abf866c2ba29ae6f6e6932351a2d51ff38c1f29b0401a981e'
ROOT=Path('/content/spoiler-context-v1'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert all((ROOT/item.filename).resolve().is_relative_to(ROOT.resolve()) for item in z.infolist())
    z.extractall(ROOT)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','safetensors>=0.4.3'])
import torch
assert torch.cuda.is_available(), 'GPU required; no CPU fallback'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
command=[sys.executable,'-u',str(ROOT/'compare_context_models.py'),'--data',str(ROOT/'blind.jsonl'),'--metadata',str(ROOT/'metadata.json'),'--contexts',str(ROOT/'contexts.json'),'--annotations',str(ROOT/'annotations.jsonl'),'--out',str(OUT)]
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for line in process.stdout: print(line,end='',flush=True)
status=process.wait()
archive=shutil.make_archive('/content/spoiler-context-results','zip',OUT)
for path in sorted(OUT.glob('*.json'))+[Path(archive)]:
    mime='application/zip' if path.suffix=='.zip' else 'application/json'
    data=base64.b64encode(path.read_bytes()).decode()
    display(HTML(f'<a download="{path.name}" href="data:{mime};base64,{data}">Download context {path.name}</a>'))
assert status==0, 'Inference failed; caches above permit resuming'
print('CONTEXT COMPARISON COMPLETE',flush=True)
